In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score,mean_squared_error
from sklearn.model_selection import train_test_split, GroupShuffleSplit
import os

In [2]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras import backend as K
import keras_tuner

physical_devices = tf.config.list_physical_devices('GPU')
tf.config.experimental.set_visible_devices(devices=physical_devices[0], device_type='GPU')
tf.config.experimental.set_memory_growth(physical_devices[0], enable=True)

In [3]:
keras.backend.set_image_data_format('channels_last')

#### Start

##### Functions

In [4]:
# -------------------- VG 工具 (混合式需要 ψ 与 K) --------------------
def _safe_clip(x, lo, hi):
    return tf.clip_by_value(x, lo, hi)

# ---------- 安全裁剪/清洗工具 ----------
def _sanitize(x, clip=1e4):
    x = tf.where(tf.math.is_finite(x), x, tf.zeros_like(x))
    return tf.clip_by_value(x, -clip, clip)

def _clamp_vg_params(thr, ths, alpha, n, Ks):
    thr = tf.clip_by_value(thr, 0.0, 0.6)
    ths = tf.clip_by_value(ths, thr + 1e-3, 0.85)
    alpha = tf.clip_by_value(alpha, 1e-5, 1.0)
    n = tf.clip_by_value(n, 1.05, 5.0)
    Ks = tf.clip_by_value(Ks, 1e-6, 1e3)
    return thr, ths, alpha, n, Ks

@tf.function
def vg_psi_from_theta(theta, theta_r, theta_s, alpha, n):
    eps = tf.constant(1e-7, theta.dtype)
    Se = (theta - theta_r) / tf.maximum(theta_s - theta_r, eps)
    Se = tf.clip_by_value(Se, 1e-4, 1.0 - 1e-4)
    n_eff = tf.clip_by_value(n, 1.05, 5.0)
    m = 1.0 - 1.0 / n_eff
    pow_term = tf.pow(Se, -1.0 / tf.maximum(m, 0.05)) - 1.0
    pow_term = _sanitize(pow_term, clip=1e6)
    abs_h = tf.pow(tf.maximum(pow_term, eps), 1.0 / n_eff) / tf.maximum(alpha, 1e-5)
    return -_sanitize(abs_h, clip=1e6)

@tf.function
def vg_K_from_theta(theta, theta_r, theta_s, Ks, n, l_const=0.5):
    eps = tf.constant(1e-7, theta.dtype)
    Se = (theta - theta_r) / tf.maximum(theta_s - theta_r, eps)
    Se = tf.clip_by_value(Se, 1e-4, 1.0 - 1e-4)
    n_eff = tf.clip_by_value(n, 1.05, 5.0)
    m = 1.0 - 1.0 / n_eff
    lpar = tf.ones_like(theta) * tf.cast(l_const, theta.dtype)
    one_minus = tf.pow(1.0 - tf.pow(Se, 1.0 / tf.maximum(m, 0.05)), m)
    K = Ks * tf.pow(Se, lpar) * tf.pow(1.0 - one_minus, 2.0)
    return _sanitize(K, clip=1e6)

# -------------------- 深度轴差分 --------------------
def node_grad_1st(f, dz_cm):
    dz = tf.convert_to_tensor(dz_cm, f.dtype)
    left = (f[..., 1:2] - f[..., 0:1]) / dz
    center = (f[..., 2:] - f[..., :-2]) / (2.0 * dz)
    right = (f[..., -1:] - f[..., -2:-1]) / dz
    return _sanitize(tf.concat([left, center, right], axis=-1), clip=1e6)

# -------------------- RRE 残差（混合/通量形式；vg_params 5 列） --------------------
def rre_residual_mixed_theta(theta_t_pred, theta_tm1_pred, theta_top_t, vg_params, dz_cm=10.0, dt_day=1.0):
    theta_top_t = tf.stop_gradient(tf.reshape(theta_top_t, (-1, 1)))
    theta_t_full = tf.concat([theta_top_t, theta_t_pred], axis=-1)
    lhs = (theta_t_pred - theta_tm1_pred) / tf.convert_to_tensor(dt_day, theta_t_pred.dtype)

    thr, ths, alpha, n, Ks = vg_params[..., 0], vg_params[..., 1], vg_params[..., 2], vg_params[..., 3], vg_params[..., 4]
    thr, ths, alpha, n, Ks = _clamp_vg_params(thr, ths, alpha, n, Ks)
    psi_t = vg_psi_from_theta(theta_t_full, thr, ths, alpha, n)
    K_t = vg_K_from_theta(theta_t_full, thr, ths, Ks, n, l_const=0.5)
    dpsi_dz = node_grad_1st(psi_t, dz_cm)
    q_t = _sanitize(K_t * (dpsi_dz - 1.0), clip=1e6)
    rhs = node_grad_1st(q_t, dz_cm)[..., 1:]
    return _sanitize(lhs - rhs, clip=1e6)

# -------------------- 损失函数 --------------------
def rmse_on_tensor(x, mask=None, eps=1e-12):
    if mask is not None:
        mask = tf.cast(mask, x.dtype)
        return tf.sqrt(tf.reduce_sum(tf.square(x) * mask) / (tf.reduce_sum(mask) + 1e-8) + eps)
    return tf.sqrt(tf.reduce_mean(tf.square(x)) + eps)

def supervised_rmse(theta_pred, theta_obs, y_mask=None, eps=1e-12):
    error2 = tf.square(theta_pred - theta_obs)
    if y_mask is None:
        return tf.reduce_mean(tf.sqrt(tf.reduce_mean(error2, axis=0) + eps))
    y_mask = tf.cast(y_mask, theta_pred.dtype)
    valid_num = tf.reduce_sum(y_mask, axis=0)
    rmse_depth = tf.sqrt(tf.reduce_sum(error2 * y_mask, axis=0) / tf.maximum(valid_num, 1.0) + eps)
    valid_depth = tf.cast(valid_num > 0, theta_pred.dtype)
    return tf.reduce_sum(rmse_depth * valid_depth) / tf.maximum(tf.reduce_sum(valid_depth), 1.0)

def sm_boundary_loss(theta_pred, vg_params, delta=0.0, mask=None, eps=1e-12):
    thr, ths = vg_params[..., 0], vg_params[..., 1]
    low = tf.nn.relu((thr - delta) - theta_pred)
    high = tf.nn.relu(theta_pred - (ths + delta))
    violation2 = tf.square(low) + tf.square(high)
    if mask is not None:
        mask = tf.cast(mask, theta_pred.dtype)
        return tf.sqrt(tf.reduce_sum(violation2 * mask) / (tf.reduce_sum(mask) + 1e-8) + eps)
    return tf.sqrt(tf.reduce_mean(violation2) + eps)

"""def rre_validity_mask(y_tm1, y_t, y_mask, vg_params, margin=0.0):
    thr, ths, alpha, n, Ks = vg_params[..., 0], vg_params[..., 1], vg_params[..., 2], vg_params[..., 3], vg_params[..., 4]
    thr, ths, _, _, _ = _clamp_vg_params(thr, ths, alpha, n, Ks)
    lower = thr - tf.cast(margin, y_t.dtype)
    upper = ths + tf.cast(margin, y_t.dtype)

    obs_mask = tf.ones_like(y_t, dtype=tf.bool) if y_mask is None else tf.cast(y_mask > 0.5, tf.bool)
    in_range_t = tf.math.is_finite(y_t) & (y_t > lower) & (y_t < upper)
    in_range_tm1 = tf.math.is_finite(y_tm1) & (y_tm1 > lower) & (y_tm1 < upper)

    # 缺测深度不剔除RRE；只有已观测且超出V-G容差范围的节点才判为无效
    node_ok_t = tf.logical_not(obs_mask) | in_range_t
    node_ok_tm1 = tf.logical_not(obs_mask) | in_range_tm1

    # 与当前两次节点差分对应的逐RRE依赖范围
    mask_20 = tf.reduce_all(node_ok_t[:, 0:4], axis=1) & node_ok_tm1[:, 1]
    mask_30 = tf.reduce_all(node_ok_t[:, 0:5], axis=1) & node_ok_tm1[:, 2]
    mask_40 = tf.reduce_all(node_ok_t[:, 1:5], axis=1) & node_ok_tm1[:, 3]
    mask_50 = tf.reduce_all(node_ok_t[:, 2:5], axis=1) & node_ok_tm1[:, 4]
    return tf.cast(tf.stack([mask_20, mask_30, mask_40, mask_50], axis=1), y_t.dtype)
"""


def rre_validity_mask(y_tm1, y_t, y_mask, vg_params, margin=0.05):
    thr, ths, alpha, n, Ks = vg_params[..., 0], vg_params[..., 1], vg_params[..., 2], vg_params[..., 3], vg_params[..., 4]
    thr, ths, _, _, _ = _clamp_vg_params(thr, ths, alpha, n, Ks)

    lower = thr - tf.cast(margin, y_t.dtype)
    upper = ths + tf.cast(margin, y_t.dtype)

    if y_mask is None:
        obs_mask = tf.ones_like(y_t, dtype=tf.bool)
    else:
        obs_mask = tf.cast(y_mask > 0.5, tf.bool)

    # 要求10、20、30、40、50 cm五层全部有观测
    all_obs = tf.reduce_all(obs_mask, axis=1)

    in_range_t = tf.math.is_finite(y_t) & (y_t > lower) & (y_t < upper)
    in_range_tm1 = tf.math.is_finite(y_tm1) & (y_tm1 > lower) & (y_tm1 < upper)

    # 所有层都有观测后，不再需要把缺测节点视为有效
    node_ok_t = in_range_t
    node_ok_tm1 = in_range_tm1

    mask_20 = all_obs & tf.reduce_all(node_ok_t[:, 0:4], axis=1) & node_ok_tm1[:, 1]
    mask_30 = all_obs & tf.reduce_all(node_ok_t[:, 0:5], axis=1) & node_ok_tm1[:, 2]
    mask_40 = all_obs & tf.reduce_all(node_ok_t[:, 1:5], axis=1) & node_ok_tm1[:, 3]
    mask_50 = all_obs & tf.reduce_all(node_ok_t[:, 2:5], axis=1) & node_ok_tm1[:, 4]

    return tf.cast(tf.stack([mask_20, mask_30, mask_40, mask_50], axis=1), y_t.dtype)


def boundary_validity_mask(y_obs, y_mask, vg_params, margin=0.0):
    thr, ths, alpha, n, Ks = vg_params[..., 0], vg_params[..., 1], vg_params[..., 2], vg_params[..., 3], vg_params[..., 4]
    thr, ths, _, _, _ = _clamp_vg_params(thr, ths, alpha, n, Ks)
    lower = thr - tf.cast(margin, y_obs.dtype)
    upper = ths + tf.cast(margin, y_obs.dtype)
    obs_mask = tf.ones_like(y_obs, dtype=tf.bool) if y_mask is None else tf.cast(y_mask > 0.5, tf.bool)
    in_range = tf.math.is_finite(y_obs) & (y_obs > lower) & (y_obs < upper)
    return tf.cast(obs_mask & in_range, y_obs.dtype)

# -------------------- 特征选择层 --------------------
@keras.utils.register_keras_serializable(package='SM')
class SelectLayer(layers.Layer):
    def __init__(self, indices, axis=-1, **kwargs):
        super().__init__(**kwargs)
        self.indices = list(indices) if isinstance(indices, (list, tuple, range)) else int(indices)
        self.axis = axis

    def call(self, x):
        return tf.gather(x, self.indices, axis=self.axis)

    def get_config(self):
        config = super().get_config()
        config.update({'indices': self.indices, 'axis': self.axis})
        return config

# -------------------- 模型骨干：动态LSTM + 垂向卷积 + 深度多头 --------------------
def build_backbone_two_frames_vertical_multihead(F_total, H1=128, H2=128, Hh=64, soil_h=32, static_h=16, vertical_h=64, head_h=32, dropout=0.1):
    if F_total != 31:
        raise ValueError('当前变量索引按照31个输入变量设置，请检查X的变量顺序。')

    dynamic_idx = [20, 22, 23, 24, 26, 27, 28, 30]
    global_static_idx = [21, 25, 29]
    X = keras.Input(shape=(5, F_total), name='X')

    # 动态变量时间编码
    X_dynamic = SelectLayer(dynamic_idx, name='select_dynamic')(X)
    H = layers.Dense(64, activation=tf.nn.gelu, name='dynamic_projection')(X_dynamic)
    H = layers.LayerNormalization(name='dynamic_norm')(H)
    H1_out = layers.LSTM(H1, return_sequences=True, dropout=dropout, name='lstm_1')(H)
    H2_out = layers.LSTM(H2, return_sequences=True, dropout=dropout, name='lstm_2')(H1_out)
    if H1 != H2:
        H1_out = layers.Dense(H2, use_bias=False, name='lstm_residual_projection')(H1_out)
    H = layers.LayerNormalization(name='lstm_residual_norm')(layers.Add(name='lstm_residual')([H1_out, H2_out]))
    H = layers.Cropping1D(cropping=(3, 0), name='last_two_steps')(H)
    H = layers.Dense(Hh, activation=tf.nn.gelu, name='time_feature')(H)
    H = layers.TimeDistributed(layers.RepeatVector(5), name='repeat_time_feature')(H)

    # 静态土壤属性编码，原始顺序为 sand(5), silt(5), clay(5), soc(5)
    X_static = SelectLayer(4, axis=1, name='static_snapshot')(X)
    soil = SelectLayer(range(20), name='select_soil_profile')(X_static)
    soil = layers.Permute((2, 1), name='soil_depth_order')(layers.Reshape((4, 5))(soil))
    soil = layers.Dense(soil_h, activation=tf.nn.gelu, name='soil_encoder')(soil)
    soil = layers.LayerNormalization(name='soil_norm')(soil)
    soil = layers.Reshape((2, 5, soil_h), name='repeat_soil')(layers.RepeatVector(2)(layers.Flatten()(soil)))

    # DEM、slope和LC作为全局静态信息
    static = SelectLayer(global_static_idx, name='select_global_static')(X_static)
    static = layers.Dense(static_h, activation=tf.nn.gelu, name='global_static_encoder')(static)
    static = layers.Reshape((2, 5, static_h), name='repeat_global_static')(layers.RepeatVector(10)(static))

    # 构建深度特征并沿深度方向传递相邻层信息
    Z = layers.Concatenate(axis=-1, name='feature_fusion')([H, soil, static])
    Z = layers.Dense(vertical_h, activation=tf.nn.gelu, name='vertical_projection')(Z)
    V = layers.Conv2D(vertical_h, kernel_size=(1, 5), padding='same', activation=tf.nn.gelu, name='vertical_conv_1')(Z)
    V = layers.Dropout(dropout)(V)
    V = layers.LayerNormalization(name='vertical_norm_1')(layers.Add()([Z, V]))
    V2 = layers.Conv2D(vertical_h, kernel_size=(1, 5), padding='same', activation=tf.nn.gelu, name='vertical_conv_2')(V)
    V2 = layers.Dropout(dropout)(V2)
    V = layers.LayerNormalization(name='vertical_norm_2')(layers.Add()([V, V2]))

    # 每个深度使用独立Dense预测头
    outputs = []
    for i, depth in enumerate([10, 20, 30, 40, 50]):
        Z_depth = SelectLayer(i, axis=2, name='select_%dcm' % depth)(V)
        Z_depth = layers.Dense(head_h, activation=tf.nn.gelu, name='head_%dcm_dense' % depth)(Z_depth)
        Z_depth = layers.LayerNormalization(name='head_%dcm_norm' % depth)(Z_depth)
        Z_depth = layers.Dropout(dropout, name='head_%dcm_dropout' % depth)(Z_depth)
        outputs.append(layers.Dense(1, activation='sigmoid', name='theta_%dcm' % depth)(Z_depth))

    out = layers.Concatenate(axis=-1, name='theta_2frames')(outputs)
    return keras.Model(X, out, name='LSTM_vertical_multihead')

def build_infer_from_backbone(backbone: keras.Model):
    theta_today = SelectLayer(1, axis=1, name='theta_today')(backbone.output)
    return keras.Model(backbone.inputs, theta_today, name='inference_only')


##### Model Class
对t-1也进行监督。RRE仅在观测剖面与带容差的V-G范围兼容时计算，边界损失仅在对应观测与V-G范围兼容时计算。


In [5]:
# -------------------- 自定义训练容器（RRE mixed） --------------------
class PGDLTrainer(keras.Model):
    def __init__(self, backbone, dz_cm=10.0, dt_day=1.0, use_learnable_weights=False, beta_sup_tm1=0.5, lambda_sup=0.7, lambda_phys=0.2, lambda_bc=0.1, bc_delta=0.0, vg_margin=0.0, **kwargs):
        super().__init__(**kwargs)
        self.backbone = backbone
        self.dz_cm = dz_cm
        self.dt_day = dt_day
        self.use_learnable_weights = use_learnable_weights
        self.beta_sup_tm1 = beta_sup_tm1
        self.lambda_sup = lambda_sup
        self.lambda_phys = lambda_phys
        self.lambda_bc = lambda_bc
        self.bc_delta = bc_delta
        self.vg_margin = vg_margin

        self.metric_sup = keras.metrics.Mean(name='sup_rmse')
        self.metric_phys = keras.metrics.Mean(name='phys_rmse')
        self.metric_bc = keras.metrics.Mean(name='bc_rmse')
        self.metric_total = keras.metrics.Mean(name='loss')

        if self.use_learnable_weights:
            self.raw_w_sup = tf.Variable(self.lambda_sup, trainable=True, dtype=tf.float32, name='raw_w_sup')
            self.raw_w_phys = tf.Variable(self.lambda_phys, trainable=True, dtype=tf.float32, name='raw_w_phys')
            self.raw_w_bc = tf.Variable(self.lambda_bc, trainable=True, dtype=tf.float32, name='raw_w_bc')

    @property
    def metrics(self):
        return [self.metric_total, self.metric_sup, self.metric_phys, self.metric_bc]

    def call(self, inputs, training=None):
        return self.backbone(inputs, training=training)

    def _get_task_weights(self):
        if not self.use_learnable_weights:
            return tf.constant(self.lambda_sup, tf.float32), tf.constant(self.lambda_phys, tf.float32), tf.constant(self.lambda_bc, tf.float32)
        w_sup = tf.nn.softplus(self.raw_w_sup)
        w_phys = tf.nn.softplus(self.raw_w_phys)
        w_bc = tf.nn.softplus(self.raw_w_bc)
        weight_sum = w_sup + w_phys + w_bc + 1e-8
        return w_sup / weight_sum, w_phys / weight_sum, w_bc / weight_sum

    def _compute_losses(self, inputs, y_tm1, y_t, training=False):
        X = inputs['X']
        theta_top = inputs['theta_top_t']
        vg_params = inputs['vg_params']
        y_mask_t = inputs.get('y_mask_t', None)

        y2 = self.backbone(X, training=training)
        theta_tm1_full = y2[:, 0, :]
        theta_t_full = y2[:, 1, :]

        L_sup_tm1 = supervised_rmse(theta_tm1_full, y_tm1, y_mask_t)
        L_sup_t = supervised_rmse(theta_t_full, y_t, y_mask_t)
        L_sup = self.beta_sup_tm1 * L_sup_tm1 + (1.0 - self.beta_sup_tm1) * L_sup_t

        if not self.use_learnable_weights and self.lambda_phys == 0:
            L_phys = tf.constant(0.0, dtype=theta_t_full.dtype)
        else:
            resid = rre_residual_mixed_theta(theta_t_full[:, 1:], theta_tm1_full[:, 1:], theta_top, vg_params, dz_cm=self.dz_cm, dt_day=self.dt_day)
            phys_mask = rre_validity_mask(y_tm1, y_t, y_mask_t, vg_params, margin=self.vg_margin)
            L_phys = rmse_on_tensor(resid, phys_mask)

        if not self.use_learnable_weights and self.lambda_bc == 0:
            L_bc = tf.constant(0.0, dtype=theta_t_full.dtype)
        else:
            bc_mask_tm1 = boundary_validity_mask(y_tm1, y_mask_t, vg_params, margin=self.vg_margin)
            bc_mask_t = boundary_validity_mask(y_t, y_mask_t, vg_params, margin=self.vg_margin)
            L_bc_tm1 = sm_boundary_loss(theta_tm1_full, vg_params, delta=self.bc_delta, mask=bc_mask_tm1)
            L_bc_t = sm_boundary_loss(theta_t_full, vg_params, delta=self.bc_delta, mask=bc_mask_t)
            L_bc = 0.5 * (L_bc_tm1 + L_bc_t)

        w_sup, w_phys, w_bc = self._get_task_weights()
        L_total = w_sup * L_sup + w_phys * L_phys + w_bc * L_bc
        return L_total, L_sup, L_phys, L_bc

    def train_step(self, data):
        inputs, (y_tm1, y_t) = data
        with tf.GradientTape() as tape:
            L_total, L_sup, L_phys, L_bc = self._compute_losses(inputs, y_tm1, y_t, training=True)
        grads = tape.gradient(L_total, self.trainable_variables)
        self.optimizer.apply_gradients(zip(grads, self.trainable_variables))
        self.metric_total.update_state(L_total)
        self.metric_sup.update_state(L_sup)
        self.metric_phys.update_state(L_phys)
        self.metric_bc.update_state(L_bc)
        return {m.name: m.result() for m in self.metrics}

    def test_step(self, data):
        inputs, (y_tm1, y_t) = data
        L_total, L_sup, L_phys, L_bc = self._compute_losses(inputs, y_tm1, y_t, training=False)
        self.metric_total.update_state(L_total)
        self.metric_sup.update_state(L_sup)
        self.metric_phys.update_state(L_phys)
        self.metric_bc.update_state(L_bc)
        return {m.name: m.result() for m in self.metrics}


##### Prepare Data

In [9]:
def pack_y_from_mask(ySparse,yMask):
    y_full = np.zeros(yMask.shape, dtype=np.float32)
    yMask = np.asarray(yMask, dtype=np.float32)
    y_full[np.where(yMask)] = ySparse.reshape(-1)
    assert yMask.sum() == ySparse.size #"y_sparse 与 mask 中 1 的数量不匹配，这种情况下是因为少了sm top层"
    return y_full

def extend_y_with_top(y, yMask, sm_top):
    """
    y:      [N,4]  ->  20,30,40,50 cm 的 SM
    yMask:  [N,4]  ->  20,30,40,50 cm 的 mask (0/1)
    sm_top: [N] 或 [N,1] -> 10 cm 的 SM，始终有值

    返回:
      y_5:      [N,5] -> [10,20,30,40,50]
      yMask_5:  [N,5] -> [1, mask_20, mask_30, mask_40, mask_50]
    """
    y      = np.asarray(y, dtype=np.float32)
    yMask  = np.asarray(yMask, dtype=np.float32)
    sm_top = np.asarray(sm_top, dtype=np.float32)

    # 保证 sm_top 是 (N,1)
    if sm_top.ndim == 1:
        sm_top = sm_top[:, None]           # (N,) -> (N,1)
    elif sm_top.shape[1] != 1:
        raise ValueError(f"sm_top 形状不对: {sm_top.shape} (期望 [N] 或 [N,1])")

    N = y.shape[0]
    if y.shape[1] != 4 or yMask.shape != y.shape:
        raise ValueError(f"y/yMask 形状不对: y={y.shape}, yMask={yMask.shape}，应为 [N,4]")

    # 拼接 10cm + 20-50cm
    y_5     = np.concatenate([sm_top, y], axis=1)               # [N,5]
    mask_top= np.ones((N, 1), dtype=np.float32)                 # 10cm 始终有值
    yMask_5 = np.concatenate([mask_top, yMask], axis=1)         # [N,5]

    return y_5, yMask_5



def ReadModelData(root_path: str,station_names: list,return_meta: bool = False,group_prefix: str = "",):
    """
    读取多个站点，并保持每个样本对应的站点编号和站内时间顺序。

    return_meta=False：保持原来的 6 个返回值，兼容后续测试代码。
    return_meta=True ：额外返回 station_id 和 time_index：
        station_id[n] 表示第 n 个样本来自哪个站点；
        time_index[n] 表示该样本在站点文件中的顺序，默认文件内样本已按时间升序。
    """
    X_all_tmp = []
    y_all_tmp = []
    y_all_tm1_tmp = []
    sm_top_all_tmp = []
    VG_all_tmp = []
    yMask_all_tmp = []
    station_id_tmp = []
    time_index_tmp = []

    failed_stations = []

    for station in station_names:
        try:
            X_station = np.load(os.path.join(root_path, f"{station}_X.npy"))
            VG_station = np.load(os.path.join(root_path, f"{station}_VG.npy"))
            sm_top_station = np.load(os.path.join(root_path, f"{station}_smTop.npy"))
            sm_top_tm1_station = np.load(os.path.join(root_path, f"{station}_smTopBefore.npy"))
            y_station = np.load(os.path.join(root_path, f"{station}_Y.npy"))
            y_mask_station = np.load(os.path.join(root_path, f"{station}_yMask.npy"))
            y_tm1_station = np.load(os.path.join(root_path, f"{station}_yBefore.npy"))

            y_station = pack_y_from_mask(y_station, y_mask_station)
            y_tm1_station = pack_y_from_mask(y_tm1_station, y_mask_station)

            y_5, y_mask_5 = extend_y_with_top(y_station,y_mask_station,sm_top_station)
            y_5_tm1, _ = extend_y_with_top(y_tm1_station,y_mask_station,sm_top_tm1_station)

            n_station = X_station.shape[0]
            arrays_to_check = {
                "y_t": y_5,
                "y_tm1": y_5_tm1,
                "sm_top": np.asarray(sm_top_station).reshape(-1, 1),
                "VG": VG_station,
                "yMask": y_mask_5,
            }
            for name, arr in arrays_to_check.items():
                if arr.shape[0] != n_station:
                    raise ValueError(
                        f"{station}: X 有 {n_station} 个样本，但 {name} 有 {arr.shape[0]} 个样本"
                    )

            X_all_tmp.append(X_station)
            y_all_tm1_tmp.append(y_5_tm1)
            y_all_tmp.append(y_5)
            sm_top_all_tmp.append(sm_top_station)
            VG_all_tmp.append(VG_station)
            yMask_all_tmp.append(y_mask_5)

            group_name = f"{group_prefix}:{station}" if group_prefix else station
            station_id_tmp.append(np.repeat(group_name, n_station))
            time_index_tmp.append(np.arange(n_station, dtype=np.int64))

        except Exception as exc:
            failed_stations.append((station, repr(exc)))
            print(f"读取失败: {station}: {exc}")

    if not X_all_tmp:
        raise RuntimeError(f"{root_path} 中没有成功读取任何站点")

    X_all_tmp = np.concatenate(X_all_tmp).astype(np.float32)
    y_all_tm1_tmp = np.concatenate(y_all_tm1_tmp).astype(np.float32)
    y_all_tmp = np.concatenate(y_all_tmp).astype(np.float32)
    sm_top_all_tmp = np.concatenate(sm_top_all_tmp).reshape(-1, 1).astype(np.float32)
    VG_all_tmp = np.concatenate(VG_all_tmp).astype(np.float32)
    yMask_all_tmp = np.concatenate(yMask_all_tmp).astype(np.float32)

    if failed_stations:
        print(f"成功读取 {len(station_id_tmp)} 个站点，失败 {len(failed_stations)} 个站点")

    if return_meta:
        station_id_all = np.concatenate(station_id_tmp)
        time_index_all = np.concatenate(time_index_tmp)
        return (
            X_all_tmp,
            y_all_tm1_tmp,
            y_all_tmp,
            sm_top_all_tmp,
            VG_all_tmp,
            yMask_all_tmp,
            station_id_all,
            time_index_all,
        )

    return (
        X_all_tmp,
        y_all_tm1_tmp,
        y_all_tmp,
        sm_top_all_tmp,
        VG_all_tmp,
        yMask_all_tmp,
    )


##### Read Train data

In [10]:
def SelectISMNData(X_all:np.array,y_all_tm1:np.array,y_all:np.array,sm_top_all:np.array,VG_all:np.array,yMask_all:np.array,num_depth:int):
    # yMask: shape [N, 5]，元素为 0/1 或 False/True 都可以
    # 统计每个样本有多少个深度有观测
    valid_depth_count = np.sum(yMask_all, axis=1)  # 形状 [N]

    # 条件：至少有 3 个深度有观测
    keep_idx = valid_depth_count >= num_depth         # 形状 [N]，布尔索引

    X_all = X_all[keep_idx]
    y_all_tm1 = y_all_tm1[keep_idx]
    y_all = y_all[keep_idx]
    sm_top_all = sm_top_all[keep_idx]
    VG_all = VG_all[keep_idx]
    yMask_all = yMask_all[keep_idx] 
    return X_all,y_all_tm1,y_all,sm_top_all,VG_all,yMask_all


In [8]:
root_path_train = 'F:/SM_RootZone_G/data_model/LSTM/ISMN_Train/'
station_name_ISMN = sorted(list(set(['_'.join(i.split('_')[:2]) for i in os.listdir(root_path_train)])))

X_all_ISMN,y_all_tm1_ISMN,y_all_ISMN,sm_top_all_ISMN,VG_all_ISMN,yMask_all_ISMN,station_id_ISMN,time_index_ISMN = ReadModelData(root_path_train,station_name_ISMN,return_meta=True,group_prefix='ISMN')


root_path_train = 'F:/SM_RootZone_G/data_model/LSTM/China_sm_Train/'
station_name_China = sorted(list(set(['_'.join(i.split('_')[:1]) for i in os.listdir(root_path_train)])))

X_all_ChinaSM,y_all_tm1_ChinaSM,y_all_ChinaSM,sm_top_all_ChinaSM,VG_all_ChinaSM,yMask_all_ChinaSM,station_id_ChinaSM,time_index_ChinaSM = ReadModelData(root_path_train,station_name_China,return_meta=True,group_prefix='ChinaSM')

print('ISMN', X_all_ISMN.shape,y_all_tm1_ChinaSM.shape,y_all_ChinaSM.shape,np.unique(station_id_ISMN).size)
print('China SM',X_all_ChinaSM.shape,y_all_tm1_ISMN.shape,y_all_ISMN.shape,np.unique(station_id_ChinaSM).size)




ISMN (591415, 5, 31) (2481474, 5) (2481474, 5) 489
China SM (2481474, 5, 31) (591415, 5) (591415, 5) 1850


In [9]:
minimum_depth = 3
keep_idx_ISMN = np.sum(yMask_all_ISMN, axis=1) >= minimum_depth

X_all_ISMN = X_all_ISMN[keep_idx_ISMN]
y_all_tm1_ISMN = y_all_tm1_ISMN[keep_idx_ISMN]
y_all_ISMN = y_all_ISMN[keep_idx_ISMN]
sm_top_all_ISMN = sm_top_all_ISMN[keep_idx_ISMN]
VG_all_ISMN = VG_all_ISMN[keep_idx_ISMN]
yMask_all_ISMN = yMask_all_ISMN[keep_idx_ISMN]
station_id_ISMN = station_id_ISMN[keep_idx_ISMN]
time_index_ISMN = time_index_ISMN[keep_idx_ISMN]

print('ISMN', X_all_ISMN.shape, y_all_tm1_ISMN.shape, y_all_ISMN.shape)
print('China SM', X_all_ChinaSM.shape, y_all_tm1_ChinaSM.shape, y_all_ChinaSM.shape)

ISMN (591415, 5, 31) (591415, 5) (591415, 5)
China SM (2481474, 5, 31) (2481474, 5) (2481474, 5)


In [10]:
X_all = np.concatenate([X_all_ChinaSM, X_all_ISMN],axis=0) #X_all_ChinaSM#
y_all_tm1 = np.concatenate([y_all_tm1_ChinaSM, y_all_tm1_ISMN],axis=0) #y_all_tm1_ChinaSM#
y_all = np.concatenate([y_all_ChinaSM, y_all_ISMN],axis=0) #y_all_ChinaSM#
sm_top_all = np.concatenate([sm_top_all_ChinaSM, sm_top_all_ISMN],axis=0) #sm_top_all_ChinaSM
VG_all = np.concatenate([VG_all_ChinaSM, VG_all_ISMN],axis=0) #VG_all_ChinaSM#
yMask_all = np.concatenate([yMask_all_ChinaSM, yMask_all_ISMN],axis=0) #yMask_all_ChinaSM#
station_id_all = np.concatenate([station_id_ChinaSM, station_id_ISMN])
time_index_all = np.concatenate([time_index_ChinaSM, time_index_ISMN])

In [ ]:
def split_by_station(station_ids, val_frac=0.30, seed=42):
    """
    按完整站点划分：同一站点的所有时间样本只能出现在 train 或 val 中。
    用于评价模型对“未见站点”的空间泛化能力。
    """
    station_ids = np.asarray(station_ids)
    unique_stations = np.unique(station_ids)
    if unique_stations.size < 2:
        raise ValueError('按站点划分至少需要 2 个站点')

    splitter = GroupShuffleSplit(
        n_splits=1,
        test_size=val_frac,
        random_state=seed,
    )
    train_idx, val_idx = next(
        splitter.split(
            X=np.zeros(station_ids.shape[0], dtype=np.uint8),
            groups=station_ids,
        )
    )

    train_stations = np.unique(station_ids[train_idx])
    val_stations = np.unique(station_ids[val_idx])
    overlap = np.intersect1d(train_stations, val_stations)
    if overlap.size:
        raise RuntimeError(f'训练和验证站点发生重叠: {overlap[:10]}')

    return np.sort(train_idx), np.sort(val_idx)


def split_by_time_block(
    station_ids,
    time_indices,
    val_frac=0.30,
    gap=4,
    min_train_samples=1,
    min_val_samples=1,
):
    """
    对每个站点按时间顺序切分：前段训练，最后连续一段验证。

    gap：训练段末尾与验证段起点之间丢弃的样本数。
    当前输入窗口含 5 个连续时刻，因此默认 gap=4，避免边界附近的
    训练窗口与验证窗口共享原始日期。

    用于评价“同一站点未来时段”的时间泛化能力。
    前提：每个站点 npy 文件中的样本顺序已经按时间升序排列。
    """
    station_ids = np.asarray(station_ids)
    time_indices = np.asarray(time_indices)
    if station_ids.shape != time_indices.shape:
        raise ValueError('station_ids 和 time_indices 形状必须一致')
    if not 0 < val_frac < 1:
        raise ValueError('val_frac 必须位于 (0, 1)')
    if gap < 0:
        raise ValueError('gap 不能为负数')

    train_parts = []
    val_parts = []
    skipped = []

    for station in np.unique(station_ids):
        idx = np.flatnonzero(station_ids == station)
        order = idx[np.argsort(time_indices[idx], kind='stable')]
        n = order.size

        n_val = max(min_val_samples, int(np.ceil(n * val_frac)))
        split_pos = n - n_val
        train_end = split_pos - gap

        if train_end < min_train_samples or n_val < min_val_samples:
            skipped.append((station, n))
            continue

        train_parts.append(order[:train_end])
        val_parts.append(order[split_pos:])

    if not train_parts or not val_parts:
        raise ValueError(
            '没有站点能够满足时间块划分条件；请减小 val_frac/gap，或检查每站样本数'
        )

    if skipped:
        print(f'时间块划分跳过 {len(skipped)} 个样本过少的站点，例如: {skipped[:5]}')

    train_idx = np.sort(np.concatenate(train_parts))
    val_idx = np.sort(np.concatenate(val_parts))

    if np.intersect1d(train_idx, val_idx).size:
        raise RuntimeError('train_idx 与 val_idx 发生重叠')

    return train_idx, val_idx


# ================================================================
# 选择验证集划分方式
# ================================================================
SPLIT_MODE = 'station'  # 可选：'station' 或 'time_block'
VAL_FRAC = 0.30
SPLIT_SEED = 42
TIME_GAP = X_all.shape[1] - 1  # 5 日输入窗口时为 4

if SPLIT_MODE == 'station':
    train_idx, val_idx = split_by_station(
        station_id_all,
        val_frac=VAL_FRAC,
        seed=SPLIT_SEED,
    )
elif SPLIT_MODE == 'time_block':
    train_idx, val_idx = split_by_time_block(
        station_id_all,
        time_index_all,
        val_frac=VAL_FRAC,
        gap=TIME_GAP,
    )
else:
    raise ValueError(f'未知 SPLIT_MODE: {SPLIT_MODE}')

# —— 索引子集 ——
X_train = X_all[train_idx]
y_train_tm1 = y_all_tm1[train_idx]
y_train = y_all[train_idx]
sm_top_train = sm_top_all[train_idx]
VG_train = VG_all[train_idx]
yMask_train = yMask_all[train_idx]

X_val = X_all[val_idx]
y_val_tm1 = y_all_tm1[val_idx]
y_val = y_all[val_idx]
sm_top_val = sm_top_all[val_idx]
VG_val = VG_all[val_idx]
yMask_val = yMask_all[val_idx]

train_stations = np.unique(station_id_all[train_idx])
val_stations = np.unique(station_id_all[val_idx])

print('Split mode:', SPLIT_MODE)
print(f'Train samples: {len(train_idx):,} ({len(train_idx)/len(X_all):.2%})')
print(f'Val samples  : {len(val_idx):,} ({len(val_idx)/len(X_all):.2%})')
print('Train stations:', train_stations.size)
print('Val stations  :', val_stations.size)

if SPLIT_MODE == 'station':
    station_overlap = np.intersect1d(train_stations, val_stations)
    print('Station overlap:', station_overlap.size)
    assert station_overlap.size == 0
else:
    # 时间块模式下站点应同时出现在训练和验证中，但样本索引不能重叠。
    print('Stations present in both sets:', np.intersect1d(train_stations, val_stations).size)

print('All shapes:', X_all.shape, y_all_tm1.shape, y_all.shape, sm_top_all.shape, VG_all.shape, yMask_all.shape)
print('Train shapes:', X_train.shape, y_train_tm1.shape, y_train.shape, sm_top_train.shape, VG_train.shape, yMask_train.shape)
print('Val shapes:', X_val.shape, y_val_tm1.shape, y_val.shape, sm_top_val.shape, VG_val.shape, yMask_val.shape)


In [11]:
def random_split_indices(N, val_frac=0.2, seed=42):
    rng = np.random.default_rng(seed)
    perm = rng.permutation(N)
    n_val = int(np.ceil(N * val_frac))
    val_idx = perm[:n_val]
    train_idx = perm[n_val:]
    return train_idx, val_idx


Ntr = X_all.shape[0]
train_idx, val_idx = random_split_indices(Ntr, val_frac=0.3, seed=42)

# —— 索引子集 —— 
X_train = X_all[train_idx]
y_train_tm1 = y_all_tm1[train_idx]
y_train = y_all[train_idx]
sm_top_train = sm_top_all[train_idx]
VG_train = VG_all[train_idx]
yMask_train = yMask_all[train_idx]

X_val = X_all[val_idx]
y_val_tm1 = y_all_tm1[val_idx]
y_val = y_all[val_idx]
sm_top_val = sm_top_all[val_idx]
VG_val = VG_all[val_idx]
yMask_val = yMask_all[val_idx]

print(X_all.shape,y_all_tm1.shape,y_all.shape,sm_top_all.shape,VG_all.shape,yMask_all.shape)
print(X_train.shape,y_train_tm1.shape,y_train.shape,sm_top_train.shape,VG_train.shape,yMask_train.shape)
print(X_val.shape,y_val_tm1.shape,y_val.shape,sm_top_val.shape,VG_val.shape,yMask_val.shape)

(3072889, 5, 31) (3072889, 5) (3072889, 5) (3072889, 1) (3072889, 5, 5) (3072889, 5)
(2151022, 5, 31) (2151022, 5) (2151022, 5) (2151022, 1) (2151022, 5, 5) (2151022, 5)
(921867, 5, 31) (921867, 5) (921867, 5) (921867, 1) (921867, 5, 5) (921867, 5)


In [12]:
y_all_tm1[0,:],y_all[0,:],sm_top_all[0,:],yMask_all[0,:]

(array([0.231, 0.271, 0.327, 0.313, 0.31 ], dtype=float32),
 array([0.23 , 0.271, 0.327, 0.314, 0.31 ], dtype=float32),
 array([0.23], dtype=float32),
 array([1., 1., 1., 1., 1.], dtype=float32))

##### Train

In [13]:
# -------------------- tf.data 辅助 --------------------
def make_dataset(X, y_tm1, y_t, theta_top_t, vg_params,
                 y_mask_t=None,
                 batch_size=256, shuffle=True, repeat=False):
    """
    X:           [N,5,F_total]
    y_tm1:       [N,5]   t-1 日 10/20/30/40/50 cm θ
    y_t:         [N,5]   t   日 10/20/30/40/50 cm θ
    theta_top_t: [N,1]   t 日 10 cm θ（上边界，永不缺失，用于 PDE）
    vg_params:   [N,5,5] 列=[θr,θs,α,n,Ks]，深度=[10,20,30,40,50]
    #y_mask_tm1:  [N,4]  可选，t-1 日 5 层监督 mask
    y_mask_t:    [N,5]  可选，t   日 5 层监督 mask
    """

    inputs = {
        "X": X.astype("float32"),
        "theta_top_t": theta_top_t.astype("float32"),
        "vg_params": vg_params.astype("float32"),
    }
    #if y_mask_tm1 is not None: inputs["y_mask_tm1"] = y_mask_t.astype("float32")
    if y_mask_t is not None: inputs["y_mask_t"] = y_mask_t.astype("float32")

    labels = (y_tm1.astype("float32"), y_t.astype("float32"))
    ds = tf.data.Dataset.from_tensor_slices((inputs, labels))
    if shuffle: ds = ds.shuffle(buffer_size=min(len(X), 10000), reshuffle_each_iteration=True)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    if repeat: ds = ds.repeat()
    return ds



In [14]:
def scheduler(epoch,lr):
    # 每隔30个epoch，学习率减小为原来的1/10
    if epoch % 50 == 0 and epoch != 0:
        print("lr changed to {}".format(lr * 0.1))
        return lr * 0.1
    else :
        return lr

In [25]:
# -------------------- 超参与单位 --------------------
DZ_CM = 10.0
DT_DAY = 1.0
LAMBDA_SUP = 1
LAMBDA_PHYS = 0
LAMBDA_BC = 0
VG_MARGIN = 0.05
BC_DELTA = 0.05
BETA_SUP = 0.5

F_TOTAL = X_train.shape[-1]
batch_size = 1024 * 8

train_ds = make_dataset(X_train, y_train_tm1, y_train, sm_top_train, VG_train, y_mask_t=yMask_train, batch_size=batch_size, shuffle=True)
val_ds = make_dataset(X_val, y_val_tm1, y_val, sm_top_val, VG_val, y_mask_t=yMask_val, batch_size=batch_size, shuffle=False)

backbone = build_backbone_two_frames_vertical_multihead(F_total=F_TOTAL, H1=128, H2=128, Hh=64, soil_h=32, static_h=16, vertical_h=64, head_h=32, dropout=0.1)
#backbone.summary()

static_probe = keras.Model(
    inputs=backbone.input,
    outputs=backbone.get_layer("static_snapshot").output,
)

x_probe = np.asarray(X_train[:32], dtype=np.float32)

static_snapshot = static_probe.predict(x_probe,batch_size=32,verbose=0,)

print("真实 t0 最大绝对值:",np.max(np.abs(x_probe[:, -1, :])),)
print("static_snapshot 最大绝对值:",np.max(np.abs(static_snapshot)),)
print("是否完全一致:",np.array_equal(static_snapshot,x_probe[:, -1, :],),)

np.testing.assert_allclose(static_snapshot,x_probe[:, -1, :],rtol=0,atol=0,)

static_branch_probe = keras.Model(
    inputs=backbone.input,
    outputs=[
        backbone.get_layer("static_snapshot").output,
        backbone.get_layer("select_soil_profile").output,
        backbone.get_layer("select_global_static").output,
    ],
)

snapshot, soil, global_static = static_branch_probe.predict(
    x_probe,
    batch_size=32,
    verbose=0,
)

print("snapshot max:", np.max(np.abs(snapshot)))
print("soil max:", np.max(np.abs(soil)))
print("global static max:", np.max(np.abs(global_static)))

assert not np.all(snapshot == 0)
assert not np.all(soil == 0)
assert not np.all(global_static == 0)


trainer = PGDLTrainer(backbone=backbone, dz_cm=DZ_CM, dt_day=DT_DAY, use_learnable_weights=False, beta_sup_tm1=BETA_SUP, lambda_sup=LAMBDA_SUP, lambda_phys=LAMBDA_PHYS, lambda_bc=LAMBDA_BC, bc_delta=BC_DELTA, vg_margin=VG_MARGIN)
trainer.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3, clipnorm=1.0))

save_path = 'F:/SM_RootZone_G/Model/LSTM_phyNew/Checkpoint/'
if not os.path.exists(save_path):
    os.makedirs(save_path)

callbacks = [
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=10, min_lr=1e-5, verbose=1),
    keras.callbacks.EarlyStopping(monitor='val_loss', patience=35, verbose=1, restore_best_weights=True),
    #keras.callbacks.LearningRateScheduler(scheduler),
    keras.callbacks.ModelCheckpoint(os.path.join(save_path, 'backbone_best_verticalMultiHead_NoTib_ISMN3_ChinaSM_ERA5TOP_100_SUPtm1_RRE_AllMask_kernal5_weights.h5'), save_weights_only=True, save_best_only=True, monitor='val_loss', mode='min', verbose=1)
]

history = trainer.fit(train_ds, validation_data=val_ds, epochs=500, callbacks=callbacks)


真实 t0 最大绝对值: 1801.0
static_snapshot 最大绝对值: 1801.0
是否完全一致: True
snapshot max: 1801.0
soil max: 757.95154
global static max: 1801.0
Epoch 1/500
262/263 [============================>.] - ETA: 0s - loss: 0.1020 - sup_rmse: 0.1020 - phys_rmse: 0.0000e+00 - bc_rmse: 0.0000e+00
Epoch 1: val_loss improved from inf to 0.06908, saving model to F:/SM_RootZone_G/Model/LSTM_phyNew/Checkpoint\backbone_best_verticalMultiHead_NoTib_ISMN3_ChinaSM_ERA5TOP_100_SUPtm1_RRE_AllMask_kernal5_weights.h5
263/263 [==============================] - 35s 89ms/step - loss: 0.1019 - sup_rmse: 0.1019 - phys_rmse: 0.0000e+00 - bc_rmse: 0.0000e+00 - val_loss: 0.0691 - val_sup_rmse: 0.0691 - val_phys_rmse: 0.0000e+00 - val_bc_rmse: 0.0000e+00 - lr: 0.0010
Epoch 2/500
263/263 [==============================] - ETA: 0s - loss: 0.0691 - sup_rmse: 0.0691 - phys_rmse: 0.0000e+00 - bc_rmse: 0.0000e+00
Epoch 2: val_loss improved from 0.06908 to 0.06055, saving model to F:/SM_RootZone_G/Model/LSTM_phyNew/Checkpoint\backbone_bes

In [26]:
print('Train',trainer.evaluate(train_ds,return_dict=True))
print('Val', trainer.evaluate(val_ds,return_dict=True))

263/263 [==============================] - 15s 56ms/step - loss: 0.0322 - sup_rmse: 0.0322 - phys_rmse: 0.0000e+00 - bc_rmse: 0.0000e+00
Train {'loss': 0.03221084550023079, 'sup_rmse': 0.03221084550023079, 'phys_rmse': 0.0, 'bc_rmse': 0.0}
113/113 [==============================] - 4s 37ms/step - loss: 0.0342 - sup_rmse: 0.0342 - phys_rmse: 0.0000e+00 - bc_rmse: 0.0000e+00
Val {'loss': 0.03418274596333504, 'sup_rmse': 0.03418274596333504, 'phys_rmse': 0.0, 'bc_rmse': 0.0}


##### Save Model

In [27]:
# ---- 4) 保存骨干与推断模型（结构+权重）或仅权重 ----
save_path = 'F:/SM_RootZone_G/Model/LSTM_phyNew/Trained/'
if not os.path.exists(save_path):
    os.makedirs(save_path)

model_name = 'verticalMultiHead_NoTib_ISMN3_ChinaSM_ERA5TOP_100_SUPtm1_RRE_AllMask_kernal5'
backbone.save(os.path.join(save_path, 'backbone_' + model_name + '.keras'))
infer_model = build_infer_from_backbone(backbone)
infer_model.save(os.path.join(save_path, 'infer_model_' + model_name + '.keras'))
backbone.save_weights(os.path.join(save_path, 'backbone_' + model_name + '_weights.h5'))
infer_model.save_weights(os.path.join(save_path, 'infer_model_' + model_name + '_weights.h5'))


##### Test set

In [11]:
def sm_stats(y, y_pred, yMask=None):
    """
    输入：
        y      : 观测值，任意 shape，可为 list / np.array
        y_pred : 预测值，shape 与 y 可广播或一致
        yMask  : 掩膜，与 y 可广播：
                 1 / True 表示该位置有数据（参与统计）
                 0 / False 表示该位置无数据（不参与统计）

    输出：
        R      : Pearson 相关系数 (用 np.corrcoef)
        rmse   : 均方根误差
        bias   : 平均偏差 (y_pred - y)
        ubrmse : unbiased RMSE = sqrt(RMSE^2 - bias^2)
    """
    y = np.asarray(y, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    # 如果给了 yMask，先广播到 y 的形状，然后展平
    if yMask is not None:
        yMask = np.asarray(yMask)
        try:
            yMask_b = np.broadcast_to(yMask, y.shape)
        except ValueError:
            raise ValueError("yMask 的 shape 无法与 y 进行广播，请检查输入。")
        mask_flat = yMask_b.astype(bool).ravel()
    else:
        mask_flat = None

    # 展平
    y_flat = y.ravel()
    y_pred_flat = y_pred.ravel()

    # 有限值的过滤
    valid = np.isfinite(y_flat) & np.isfinite(y_pred_flat)

    # 再叠加 yMask 的过滤（如果提供了）
    if mask_flat is not None:
        valid &= mask_flat

    yv = y_flat[valid]
    pv = y_pred_flat[valid]

    if yv.size == 0:
        return np.nan, np.nan, np.nan, np.nan

    # 误差
    diff = pv - yv

    # RMSE
    rmse = np.sqrt(np.mean(diff**2))

    # bias
    bias = np.mean(diff)

    # ubRMSE
    ubrmse = np.sqrt(max(rmse**2 - bias**2, 0.0))

    # 相关系数 R，用 np.corrcoef
    if yv.size < 2:
        R = np.nan
    else:
        R = np.corrcoef(yv, pv)[0, 1]

    return R, rmse, bias, ubrmse

def getMetrics(model,root_path,station_names:list,batch_size=2048,mode='LSTM',depth_idx=0):
    sta = []
    r = []
    rmse = []
    bias = []
    ubrmse = []
    depth = []
    for station in station_names:
        
        #X_test,y_test,_,_,_,yMask_test = ReadModelData(root_path,[station])
        X_test, _, y_test, _, _, yMask_test = ReadModelData(root_path,[station])
        if mode =='ANN':
            y_pred = model.predict(X_test[:,-1,:], batch_size=batch_size, verbose=0)
            sta.append(station)
            tmp_r,tmp_rmse,tmp_bias,tmp_ubrmse = sm_stats(y_test[:,depth_idx],y_pred,yMask_test[:,depth_idx])
            r.append(tmp_r)
            rmse.append(tmp_rmse)
            bias.append(tmp_bias)
            ubrmse.append(tmp_ubrmse)
            depth.append(10*(depth_idx+1))
        else:
            y_pred = model.predict(X_test, batch_size=batch_size, verbose=0)
            for d in range(5):
                sta.append(station)
                tmp_r,tmp_rmse,tmp_bias,tmp_ubrmse = sm_stats(y_test[:,d],y_pred[:,d],yMask_test[:,d])
                r.append(tmp_r)
                rmse.append(tmp_rmse)
                bias.append(tmp_bias)
                ubrmse.append(tmp_ubrmse)
                depth.append(10*(d+1))
    df = pd.DataFrame({'Station':sta,'R':r,'ubRMSE':ubrmse,'RMSE':rmse,'Bias':bias,'Depth':depth})
    return df

In [12]:
#X_all_ISMN,y_all_tm1_ISMN,y_all_ISMN,sm_top_all_ISMN,VG_all_ISMN,yMask_all_ISMN

root_path_test = 'F:/SM_RootZone_G/data_model/LSTM/China_sm_Test/'
station_names = sorted(list(set(['_'.join(i.split('_')[:1]) for i in os.listdir(root_path_test)])))

X_test_ChinaSM,_,y_test_ChinaSM,_,_,yMask_test_ChinaSM = ReadModelData(root_path_test,station_names)

root_path_test = 'F:/SM_RootZone_G/data_model/LSTM/ISMN_Test/'
station_names = sorted(list(set(['_'.join(i.split('_')[:2]) for i in os.listdir(root_path_test)])))

X_test_ISMN,y_test_tm1_ISMN,y_test_ISMN,sm_top_test_ISMN,VG_test_ISMN,yMask_test_ISMN = ReadModelData(root_path_test,station_names)

minimum_depth = 3
X_test_ISMN,_,y_test_ISMN,_,_,yMask_test_ISMN = SelectISMNData(X_test_ISMN,y_test_tm1_ISMN,y_test_ISMN,sm_top_test_ISMN,VG_test_ISMN,yMask_test_ISMN,minimum_depth)

print('China SM Test',X_test_ChinaSM.shape,y_test_ChinaSM.shape,yMask_test_ChinaSM.shape)
print('ISMN Test',X_test_ISMN.shape)

China SM Test (611394, 5, 31) (611394, 5) (611394, 5)
ISMN Test (99141, 5, 31)


In [13]:
root_path = 'F:/SM_RootZone_G/Model/LSTM_phyNew/Trained/'
batch_size = 1024*8
# ---- 5) 预测（推断） ----
# 5.1 直接加载推断模型（最省事）
X_test = X_test_ChinaSM
y_test = y_test_ChinaSM
yMask_test = yMask_test_ChinaSM 
model_name = "infer_model_"+"verticalMultiHead_NoTib_ISMN3_ChinaSM_ERA5TOP_100_SUPtm1_RRE_AllMask_kernal5"
infer_loaded = keras.models.load_model(os.path.join(root_path,model_name+'.keras'))
infer_loaded.compile(optimizer="adam",loss="mse",metrics=[keras.metrics.RootMeanSquaredError(name="rmse")])
print(infer_loaded.evaluate(X_test,y_test,batch_size=batch_size,verbose=1))

theta_t_pred = infer_loaded.predict(X_test, batch_size=batch_size, verbose=0)  # [Nval,4]
print(theta_t_pred.shape,y_test.shape)
print('R, RMSE, Bias, ubRMSE')
print('10cm', sm_stats(y_test[:,0],theta_t_pred[:,0],yMask_test[:,0]))
print('20cm', sm_stats(y_test[:,1],theta_t_pred[:,1],yMask_test[:,1]))
print('30cm', sm_stats(y_test[:,2],theta_t_pred[:,2],yMask_test[:,2]))
print('40cm', sm_stats(y_test[:,3],theta_t_pred[:,3],yMask_test[:,3]))
print('50cm', sm_stats(y_test[:,4],theta_t_pred[:,4],yMask_test[:,4]))


75/75 [==============================] - 10s 40ms/step - loss: 0.0020 - rmse: 0.0447
[0.002002512337639928, 0.04474944248795509]
(611394, 5) (611394, 5)
R, RMSE, Bias, ubRMSE
10cm (0.9772345968047444, 0.01830573816144501, 0.000804247689047833, 0.018288062644524306)
20cm (0.9704187431164031, 0.02063484345263036, 0.0005740561374977469, 0.020626856858609436)
30cm (0.9606135378079591, 0.025775269032639273, 0.001079641423866779, 0.025752647788155766)
40cm (0.962142911814559, 0.025201396925058375, 0.0010361658848758188, 0.025180086720131868)
50cm (0.6722692008548748, 0.08917505730474588, -0.04178859685044551, 0.07877756037461206)


In [14]:
root_path = 'F:/SM_RootZone_G/Model/LSTM_phyNew/Trained/'
batch_size = 1024*8
# ---- 5) 预测（推断） ----
# 5.1 直接加载推断模型（最省事）
X_test = X_test_ISMN
y_test = y_test_ISMN
yMask_test = yMask_test_ISMN
model_name = "infer_model_"+"verticalMultiHead_NoTib_ISMN3_ChinaSM_ERA5TOP_100_SUPtm1_RRE_AllMask_kernal5"
infer_loaded = keras.models.load_model(os.path.join(root_path,model_name+'.keras'))
infer_loaded.compile(optimizer="adam",loss="mse",metrics=[keras.metrics.RootMeanSquaredError(name="rmse")])
print(infer_loaded.evaluate(X_test,y_test,batch_size=batch_size,verbose=1))

theta_t_pred = infer_loaded.predict(X_test, batch_size=batch_size, verbose=0)  # [Nval,4]
print(theta_t_pred.shape,y_test.shape)
print('R, RMSE, Bias, ubRMSE')
print('10cm', sm_stats(y_test[:,0],theta_t_pred[:,0],yMask_test[:,0]))
print('20cm', sm_stats(y_test[:,1],theta_t_pred[:,1],yMask_test[:,1]))
print('30cm', sm_stats(y_test[:,2],theta_t_pred[:,2],yMask_test[:,2]))
print('40cm', sm_stats(y_test[:,3],theta_t_pred[:,3],yMask_test[:,3]))
print('50cm', sm_stats(y_test[:,4],theta_t_pred[:,4],yMask_test[:,4]))


13/13 [==============================] - 3s 31ms/step - loss: 0.0327 - rmse: 0.1807
[0.03265916183590889, 0.18071846663951874]
(99141, 5) (99141, 5)
R, RMSE, Bias, ubRMSE
10cm (0.8748328101875769, 0.05331042602080208, -0.003327483404587735, 0.05320647870994288)
20cm (0.8815093446064349, 0.05066051251908207, -0.0006557134068456795, 0.05065626879887773)
30cm (0.8554998785104141, 0.04632243908555708, -0.025793480384966424, 0.03847680772186667)
40cm (0.7195784313239117, 0.07006327807261743, -0.023128824849282543, 0.06613562122919936)
50cm (0.8850240342303916, 0.052772551875112976, 0.0009625235824107908, 0.05276377336549001)


In [30]:
root_path = 'F:/SM_RootZone_G/Model/LSTM_phyNew/Trained/'
batch_size = 1024*8
# ---- 5) 预测（推断） ----
# 5.1 直接加载推断模型（最省事）

model_name = "infer_model_"+"verticalMultiHead_NoTib_ISMN3_ChinaSM_ERA5TOP_100_SUPtm1_RRE_AllMask_kernal5"
infer_loaded = keras.models.load_model(os.path.join(root_path,model_name+'.keras'))
infer_loaded.compile(optimizer="adam",loss="mse",metrics=[keras.metrics.RootMeanSquaredError(name="rmse")])
print(infer_loaded.evaluate(X_test,y_test,batch_size=batch_size,verbose=1))

root_path = 'F:/SM_RootZone_G/data_model/QTP/ISMN/'
station_names = sorted(list(set(['_'.join(i.split('_')[:2]) for i in os.listdir(root_path)])))
df = getMetrics(infer_loaded,root_path,station_names,batch_size)
df['Network'] = df['Station'].apply(lambda X: X.split('_')[0])
df['Station'] = df['Station'].apply(lambda X: X.split('_')[1])
print(model_name)
print(df.groupby(['Depth'])[['R','ubRMSE','RMSE','Bias']].mean())
df.groupby(['Network','Depth'])[['R','ubRMSE','RMSE','Bias']].mean()

9/9 [==============================] - 2s 30ms/step - loss: 0.0387 - rmse: 0.1967
[0.03869836404919624, 0.1967190057039261]
infer_model_verticalMultiHead_NoTib_ISMN3_ChinaSM_ERA5TOP_100_SUPtm1_RRE_AllMask_kernal5
              R    ubRMSE      RMSE      Bias
Depth                                        
10     0.517749  0.042210  0.091154  0.058190
20     0.410120  0.035145  0.146177  0.127098
30          NaN       NaN       NaN       NaN
40     0.362684  0.030556  0.176265  0.166908
50          NaN       NaN       NaN       NaN


R    ubRMSE      RMSE      Bias
Network   Depth                                        
CTP-SMTMN 10     0.503023  0.045931  0.085704  0.056449
          20     0.389419  0.034790  0.166909  0.161930
          30          NaN       NaN       NaN       NaN
          40     0.429103  0.033382  0.182307  0.176661
          50          NaN       NaN       NaN       NaN
MAQU      10     0.427362  0.047533  0.092187  0.032224
          20     0.315526  0.037769  0.081413  0.030122
          30          NaN       NaN       NaN       NaN
          40     0.160943  0.028708  0.101674  0.071475
          50          NaN       NaN       NaN       NaN
NAQU      10     0.572971  0.032724  0.122174  0.106862
          20     0.441035  0.032714  0.239436  0.236855
          30          NaN       NaN       NaN       NaN
          40     0.389446  0.021721  0.208211  0.206937
          50          NaN       NaN       NaN       NaN
NGARI     10     0.594643  0.033590  0.089061  0.058420
          20     0.516553  0.035250  0.098065  0.059958
          30          NaN       NaN       NaN       NaN
          40     0.334393  0.029145  0.201145  0.194130
          50          NaN       NaN       NaN       NaN

In [26]:
root_path = 'F:/SM_RootZone_G/Model/LSTM_phyNew/Trained/'
batch_size = 1024*8
# ---- 5) 预测（推断） ----
# 5.1 直接加载推断模型（最省事）

model_name = "infer_model_"+"verticalMultiHead_NoTib_ISMN3_ChinaSM_ERA5TOP_811_SUPtm1_RRE_AllMask_kernal5"
infer_loaded = keras.models.load_model(os.path.join(root_path,model_name+'.keras'))
infer_loaded.compile(optimizer="adam",loss="mse",metrics=[keras.metrics.RootMeanSquaredError(name="rmse")])
print(infer_loaded.evaluate(X_test,y_test,batch_size=batch_size,verbose=1))

root_path = 'F:/SM_RootZone_G/data_model/QTP/ISMN/'
station_names = sorted(list(set(['_'.join(i.split('_')[:2]) for i in os.listdir(root_path)])))
df = getMetrics(infer_loaded,root_path,station_names,batch_size)
df['Network'] = df['Station'].apply(lambda X: X.split('_')[0])
df['Station'] = df['Station'].apply(lambda X: X.split('_')[1])
print(model_name)
print(df.groupby(['Depth'])[['R','ubRMSE','RMSE','Bias']].mean())
df.groupby(['Network','Depth'])[['R','ubRMSE','RMSE','Bias']].mean()

9/9 [==============================] - 2s 29ms/step - loss: 0.0293 - rmse: 0.1711
[0.029286067932844162, 0.17113173007965088]
infer_model_verticalMultiHead_NoTib_ISMN3_ChinaSM_ERA5TOP_811_SUPtm1_RRE_AllMask_kernal5
              R    ubRMSE      RMSE      Bias
Depth                                        
10     0.471420  0.043325  0.087416 -0.044897
20     0.410771  0.032962  0.069000 -0.023909
30          NaN       NaN       NaN       NaN
40     0.308713  0.029451  0.123237  0.104843
50          NaN       NaN       NaN       NaN


R    ubRMSE      RMSE      Bias
Network   Depth                                        
CTP-SMTMN 10     0.489196  0.046798  0.095232 -0.070254
          20     0.400909  0.032814  0.065838 -0.042061
          30          NaN       NaN       NaN       NaN
          40     0.318837  0.033675  0.125814  0.116830
          50          NaN       NaN       NaN       NaN
MAQU      10     0.392906  0.049658  0.090954  0.030674
          20     0.339224  0.040825  0.081442  0.029902
          30          NaN       NaN       NaN       NaN
          40     0.211090  0.027898  0.086017  0.043882
          50          NaN       NaN       NaN       NaN
NAQU      10     0.458236  0.034103  0.066918 -0.005155
          20     0.395758  0.026227  0.056218  0.034681
          30          NaN       NaN       NaN       NaN
          40     0.315746  0.019379  0.156795  0.155347
          50          NaN       NaN       NaN       NaN
NGARI     10     0.491705  0.034448  0.075232 -0.056919
          20     0.495275  0.030737  0.073753 -0.046718
          30          NaN       NaN       NaN       NaN
          40     0.352413  0.024946  0.128313  0.096042
          50          NaN       NaN       NaN       NaN

In [43]:
pd.set_option("display.max_rows", None)
df

,Station,R,ubRMSE,RMSE,Bias,Depth,Network
0,L01,0.541430,0.042255,0.160250,0.154579,10,CTP-SMTMN
1,L01,0.442280,0.034975,0.278089,0.275881,20,CTP-SMTMN
2,L01,NaN,NaN,NaN,NaN,30,CTP-SMTMN
3,L01,0.408755,0.052014,0.235742,0.229932,40,CTP-SMTMN
4,L01,NaN,NaN,NaN,NaN,50,CTP-SMTMN
5,L02,0.426814,0.059690,0.064095,-0.023352,10,CTP-SMTMN
6,L02,0.428555,0.032392,0.140972,0.137200,20,CTP-SMTMN
7,L02,NaN,NaN,NaN,NaN,30,CTP-SMTMN
8,L02,0.723144,0.021695,0.194482,0.193268,40,CTP-SMTMN
9,L02,NaN,NaN,NaN,NaN,50,CTP-SMTMN


In [16]:
root_path_test = 'F:/SM_RootZone_G/data_model/QTP/ISMN/'
station_names = sorted(list(set(['_'.join(i.split('_')[:2]) for i in os.listdir(root_path_test)])))

X_test_QTP,_,y_test_QTP,_,_,yMask_test_QTP = ReadModelData(root_path_test,station_names)

print('ISMN QTP Test',X_test_QTP.shape)

ISMN QTP Test (73194, 5, 31)


In [19]:
root_path = 'F:/SM_RootZone_G/Model/LSTM_phyNew/Trained/'
batch_size = 1024*8
# ---- 5) 预测（推断） ----
# 5.1 直接加载推断模型（最省事）
X_test = X_test_QTP
y_test = y_test_QTP
yMask_test = yMask_test_QTP
model_name = "infer_model_"+"verticalMultiHead_NoTib_ISMN3_ChinaSM_ERA5TOP_811_SUPtm1_RRE_AllMask_kernal5"
infer_loaded = keras.models.load_model(os.path.join(root_path,model_name+'.keras'))
infer_loaded.compile(optimizer="adam",loss="mse",metrics=[keras.metrics.RootMeanSquaredError(name="rmse")])
print(infer_loaded.evaluate(X_test,y_test,batch_size=batch_size,verbose=1))

theta_t_pred = infer_loaded.predict(X_test, batch_size=batch_size, verbose=0)  # [Nval,4]
print(theta_t_pred.shape,y_test.shape)
print('R, RMSE, Bias, ubRMSE')
print('10cm', sm_stats(y_test[:,0],theta_t_pred[:,0],yMask_test[:,0]))
print('20cm', sm_stats(y_test[:,1],theta_t_pred[:,1],yMask_test[:,1]))
print('30cm', sm_stats(y_test[:,2],theta_t_pred[:,2],yMask_test[:,2]))
print('40cm', sm_stats(y_test[:,3],theta_t_pred[:,3],yMask_test[:,3]))
print('50cm', sm_stats(y_test[:,4],theta_t_pred[:,4],yMask_test[:,4]))


9/9 [==============================] - 2s 23ms/step - loss: 0.0293 - rmse: 0.1711
[0.029286067932844162, 0.17113173007965088]
(73194, 5) (73194, 5)
R, RMSE, Bias, ubRMSE
10cm (0.6130547656657975, 0.09618855325214526, -0.0476317173312534, 0.0835670825194727)
20cm (0.6742655930181333, 0.07601849693441519, -0.026284552044559695, 0.07132975676381158)
30cm (nan, nan, nan, nan)
40cm (0.376380292046817, 0.13666941949754446, 0.10707992369368209, 0.08492596874661489)
50cm (nan, nan, nan, nan)


In [29]:
root_path = 'F:/SM_RootZone_G/Model/LSTM_phyNew/Trained/'
batch_size = 1024*8
# ---- 5) 预测（推断） ----
# 5.1 直接加载推断模型（最省事）
X_test = X_test_QTP
y_test = y_test_QTP
yMask_test = yMask_test_QTP
model_name = "infer_model_"+"verticalMultiHead_NoTib_ISMN3_ChinaSM_ERA5TOP_100_SUPtm1_RRE_AllMask_kernal5"
infer_loaded = keras.models.load_model(os.path.join(root_path,model_name+'.keras'))
infer_loaded.compile(optimizer="adam",loss="mse",metrics=[keras.metrics.RootMeanSquaredError(name="rmse")])
print(infer_loaded.evaluate(X_test,y_test,batch_size=batch_size,verbose=1))

theta_t_pred = infer_loaded.predict(X_test, batch_size=batch_size, verbose=0)  # [Nval,4]
print(theta_t_pred.shape,y_test.shape)
print('R, RMSE, Bias, ubRMSE')
print('10cm', sm_stats(y_test[:,0],theta_t_pred[:,0],yMask_test[:,0]))
print('20cm', sm_stats(y_test[:,1],theta_t_pred[:,1],yMask_test[:,1]))
print('30cm', sm_stats(y_test[:,2],theta_t_pred[:,2],yMask_test[:,2]))
print('40cm', sm_stats(y_test[:,3],theta_t_pred[:,3],yMask_test[:,3]))
print('50cm', sm_stats(y_test[:,4],theta_t_pred[:,4],yMask_test[:,4]))


9/9 [==============================] - 2s 29ms/step - loss: 0.0387 - rmse: 0.1967
[0.03869836404919624, 0.1967190057039261]
(73194, 5) (73194, 5)
R, RMSE, Bias, ubRMSE
10cm (0.6159397587389691, 0.09734588534745388, 0.057362492187767004, 0.07864964007602326)
20cm (0.4019678168446022, 0.158445347130444, 0.12688308672021997, 0.09489789424236998)
30cm (nan, nan, nan, nan)
40cm (0.23248113559405065, 0.19169538009805895, 0.16930503785035023, 0.08990507721720042)
50cm (nan, nan, nan, nan)


##### RRE loss

In [ ]:
def observed_rre_loss(y_tm1, y_t, sm_top_t, vg_params, y_mask, dz_cm=DZ_CM, dt_day=DT_DAY):
    valid_idx = np.all(y_mask[:, 1:] > 0.5, axis=1)
    valid_idx &= np.all(np.isfinite(y_tm1[:, 1:]), axis=1)
    valid_idx &= np.all(np.isfinite(y_t[:, 1:]), axis=1)
    valid_idx &= np.all(np.isfinite(sm_top_t), axis=1)
    valid_idx &= np.all(np.isfinite(vg_params.reshape(vg_params.shape[0], -1)), axis=1)

    theta_tm1_obs = tf.convert_to_tensor(y_tm1[valid_idx, 1:], dtype=tf.float32)
    theta_t_obs = tf.convert_to_tensor(y_t[valid_idx, 1:], dtype=tf.float32)
    theta_top_t_obs = tf.convert_to_tensor(sm_top_t[valid_idx], dtype=tf.float32)
    vg_obs = tf.convert_to_tensor(vg_params[valid_idx], dtype=tf.float32)

    resid_obs = rre_residual_mixed_theta(theta_t_obs, theta_tm1_obs, theta_top_t_obs, vg_obs, dz_cm=dz_cm, dt_day=dt_day)

    rre_rmse = tf.sqrt(tf.reduce_mean(tf.square(resid_obs)))
    rre_rmse_depth = tf.sqrt(tf.reduce_mean(tf.square(resid_obs), axis=0))

    print('Valid samples:', np.sum(valid_idx))
    print('Observed RRE RMSE:', float(rre_rmse.numpy()))
    print('Observed RRE RMSE by depth [20, 30, 40, 50 cm]:', rre_rmse_depth.numpy())

    return resid_obs.numpy(), float(rre_rmse.numpy()), rre_rmse_depth.numpy()

In [ ]:
resid_obs_train, rre_obs_train, rre_obs_depth_train = observed_rre_loss(
    y_train_tm1, y_train, sm_top_train, VG_train, yMask_train
)

resid_obs_val, rre_obs_val, rre_obs_depth_val = observed_rre_loss(
    y_val_tm1, y_val, sm_top_val, VG_val, yMask_val
)

resid_obs_ISMN, rre_obs_ISMN, rre_obs_depth_ISMN = observed_rre_loss(
    y_all_tm1_ISMN,y_all_ISMN,sm_top_all_ISMN,VG_all_ISMN,yMask_all_ISMN
)


In [ ]:
valid_idx = np.all(yMask_train[:, 1:] > 0.5, axis=1)
theta_t = tf.convert_to_tensor(y_train[valid_idx], dtype=tf.float32)
VG_tmp = tf.convert_to_tensor(VG_train[valid_idx], dtype=tf.float32)

thr, ths = VG_tmp[..., 0], VG_tmp[..., 1]
alpha, n, Ks = VG_tmp[..., 2], VG_tmp[..., 3], VG_tmp[..., 4]
thr, ths, alpha, n, Ks = _clamp_vg_params(thr, ths, alpha, n, Ks)

Se_raw = (theta_t - thr) / (ths - thr)
psi_t = vg_psi_from_theta(theta_t, thr, ths, alpha, n)
K_t = vg_K_from_theta(theta_t, thr, ths, Ks, n)
dpsi_dz = node_grad_1st(psi_t, DZ_CM)
q_t = K_t * (dpsi_dz - 1.0)

depths = [10, 20, 30, 40, 50]

for i, depth in enumerate(depths):
    print('\n%d cm' % depth)
    print('Se outside [0,1]:', np.mean((Se_raw[:, i].numpy() <= 0) | (Se_raw[:, i].numpy() >= 1)))
    print('psi quantiles:', np.percentile(psi_t[:, i].numpy(), [0, 1, 50, 99, 100]))
    print('K quantiles:', np.percentile(K_t[:, i].numpy(), [0, 1, 50, 99, 100]))
    print('q quantiles:', np.percentile(q_t[:, i].numpy(), [0, 1, 50, 99, 100]))

In [ ]:
valid_idx = np.all(yMask_train[:, 1:] > 0.5, axis=1)
theta_t = y_train[valid_idx]
VG_tmp = VG_train[valid_idx]

thr = VG_tmp[..., 0]
ths = VG_tmp[..., 1]

depths = [10, 20, 30, 40, 50]

for i, depth in enumerate(depths):
    low_ratio = np.mean(theta_t[:, i] <= thr[:, i])
    high_ratio = np.mean(theta_t[:, i] >= ths[:, i])
    print('%d cm: theta <= theta_r: %.4f, theta >= theta_s: %.4f' % (depth, low_ratio, high_ratio))

In [ ]:
for i, depth in enumerate(depths):
    valid = yMask_train[:, i].astype(bool)

    low_tm1 = np.mean(y_train_tm1[valid, i] <= thr[valid, i])
    high_tm1 = np.mean(y_train_tm1[valid, i] >= ths[valid, i])
    low_t = np.mean(y_train[valid, i] <= thr[valid, i])
    high_t = np.mean(y_train[valid, i] >= ths[valid, i])

    print('%d cm | t-1: low %.4f, high %.4f | t: low %.4f, high %.4f' %
          (depth, low_tm1, high_tm1, low_t, high_t))

In [ ]:
valid_depth = np.all(yMask_train[:, 1:] > 0.5, axis=1)

thr = VG_train[..., 0]
ths = VG_train[..., 1]

valid_vg = valid_depth.copy()
valid_vg &= np.all((y_train_tm1 > thr) & (y_train_tm1 < ths), axis=1)
valid_vg &= np.all((y_train > thr) & (y_train < ths), axis=1)

low50 = valid_depth & ((y_train_tm1[:, 4] <= thr[:, 4]) | (y_train[:, 4] <= thr[:, 4]))
high50 = valid_depth & ((y_train_tm1[:, 4] >= ths[:, 4]) | (y_train[:, 4] >= ths[:, 4]))

In [ ]:
def calc_group_rre(idx, name):
    resid = rre_residual_mixed_theta(
        tf.convert_to_tensor(y_train[idx, 1:], dtype=tf.float32),
        tf.convert_to_tensor(y_train_tm1[idx, 1:], dtype=tf.float32),
        tf.convert_to_tensor(sm_top_train[idx], dtype=tf.float32),
        tf.convert_to_tensor(VG_train[idx], dtype=tf.float32),
        dz_cm=DZ_CM,
        dt_day=DT_DAY
    )

    total = tf.sqrt(tf.reduce_mean(tf.square(resid))).numpy()
    by_depth = tf.sqrt(tf.reduce_mean(tf.square(resid), axis=0)).numpy()

    print(name, 'N =', np.sum(idx))
    print('RRE RMSE =', total)
    print('By depth =', by_depth)

calc_group_rre(valid_vg, 'All depths within VG range')
calc_group_rre(low50, '50 cm below theta_r')
calc_group_rre(high50, '50 cm above theta_s')

#### Bottom